# Part 4: Structural identifiability analysis

Structural identifiability analysis aims to describe what parameter values (and other model properties) are fundamentally possible to determine uniquely from a specific type of data. It assumes that infinitely many noise-free data points are available, and it is different from *practical identifiability analysis*, which aims to describe what model properties can be estimated reliably in the presence of noise and limited data.

### Part 4.1: A simple exponential growth example
Let's consider the following example with exponential growth of a rabbit population ($R$). Here, rabbit growth is due to the frequency of births ($b_f$) and the number of new rabbits per birth ($b_n$). However, trivially, if we only veer can meassured $R$, we will never be able to find the specific nuber $b_f$ and $b_n$, just their product.

![alt text](assets/parameters_struct_ident_example.png)

We next consider this model, but how to determine identifiability in Julia. We declare it as a reaction system with a single reaction (a rabbit giving birth at rate `bf*bn`).


In [ ]:
using Catalyst 
rn = @reaction_network begin
    bf*bn, R --> 2R
end

We can check the corresponding ODE model:


In [ ]:
ode_model(rn)

Finally, we can determine structural identifiability using the `assess_identifiability` function. Here, we also specify which quantities we can measure.


In [ ]:
using StructuralIdentifiability
assess_identifiability(rn; measured_quantities = [:R])

Using the `find_identifiable_functions` function we can confirm that the product of the two parameters is the only quantity that we can identify from data

In [ ]:
find_identifiable_functions(gwo; measured_quantities = [:M])

### Part 4.2: A more complicated model (Goodwin oscillator)
This case was trivial; however, StructuralIdentifiability.jl can also handle much more complicated models. Let's now consider the Goodwin oscillator model:


In [ ]:
gwo = @reaction_network begin
    (pₘ/(1+P), dₘ), 0 <--> M
    (pₑ*M,dₑ), 0 <--> E
    (pₚ*E,dₚ), 0 <--> P
end
ode_model(gwo)

Here, we see that we have a combination of *globally identifiable*, *locally identifiable*, and *non-identifiable* parameters.


In [ ]:
assess_identifiability(gwo; measured_quantities = [:M])

A locally identifiable parameter is a parameter such that one can determine it down to a countable number of possible values, but not to a single value. For example, in a model where $K^2$ appears, $K$ and $-K$ yield identical dynamics, and $K$ is thus locally identifiable. If one only cares about local identifiability, the `assess_local_identifiability` function can be used (which generally computes quicker than the full identifiability assessment).


In [ ]:
assess_local_identifiability(gwo; measured_quantities = [:M])

Finally, we can use the `find_identifiable_functions` function to find a list of all possible expressions that we *can* identify:


In [ ]:
find_identifiable_functions(gwo; measured_quantities = [:M])

Here, we can see that the identifiable parameters appear as identifiable expressions, while the non-identifiable ones are entangled with other parameters.


We can also list multiple measured quantities in `measured_quantities`, which should improve identifiability.


In [ ]:
assess_identifiability(gwo; measured_quantities = [:M, :P])

What if we try another combination of measured quantities? Here, the identifiability pattern changes, and can even improve.


In [ ]:
assess_identifiability(gwo; measured_quantities = [:M, :E])

In [ ]:
assess_identifiability(gwo; measured_quantities = [:E, :P])

This is an important utility of structural identifiability analysis. By telling us how identifiability depends on what we measure, it is possible to better design experiments to maximise identifiability.
